# Orpheus TTS - Fully Fixed (CPU-Only)

## All fixes applied

| Fix | Detail |
|-----|--------|
| Prompt format | Token ID list, not string |
| SNAC offset | Per-position: `(name_num-10) - (pos*4096)` |
| Repetition | `rep_penalty=1.3` + dynamic `max_tokens` cap |


## Cell 1 - Install Dependencies

In [18]:
import subprocess, sys

def pip(pkg, extra_index=None):
    cmd = [sys.executable, '-m', 'pip', 'install', pkg, '-q']
    if extra_index:
        cmd += ['--extra-index-url', extra_index]
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(f'[{"OK" if r.returncode==0 else "FAIL"}] {pkg}')
    if r.returncode != 0: print(r.stderr[-1000:])

pip('llama-cpp-python', extra_index='https://abetlen.github.io/llama-cpp-python/whl/cpu')
pip('snac')
pip('soundfile')
pip('huggingface_hub')
print('Done.')

[OK] llama-cpp-python
[OK] snac
[OK] soundfile
[OK] huggingface_hub
Done.


## Cell 2 - Download Model

In [19]:
from huggingface_hub import hf_hub_download
import os

MODEL_REPO = 'isaiahbjork/orpheus-3b-0.1-ft-Q4_K_M-GGUF'
MODEL_FILE = 'orpheus-3b-0.1-ft-q4_k_m.gguf'
MODELS_DIR = 'models'
os.makedirs(MODELS_DIR, exist_ok=True)
model_path = os.path.join(MODELS_DIR, MODEL_FILE)

if not os.path.exists(model_path):
    print('Downloading (~2 GB)...')
    model_path = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE, local_dir=MODELS_DIR)

print(f'Model: {model_path}')

Model: models\orpheus-3b-0.1-ft-q4_k_m.gguf


## Cell 3 - Load Models

In [20]:
import torch, snac
from llama_cpp import Llama

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

print('Loading Orpheus GGUF...')
llm = Llama(model_path=model_path, n_ctx=4096, n_gpu_layers=0, verbose=False)
print('Orpheus OK')

print('Loading SNAC decoder...')
snac_model = snac.SNAC.from_pretrained('hubertsiuzdak/snac_24khz').eval().to(DEVICE)
print('SNAC OK')

Device: cpu
Loading Orpheus GGUF...


llama_context: n_ctx_seq (4096) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


Orpheus OK
Loading SNAC decoder...
SNAC OK


## Cell 4 - TTS Functions

In [21]:
import re
import numpy as np
import soundfile as sf

POS_OFFSETS = [i * 4096 for i in range(7)]

def calc_max_tokens(text: str) -> int:
    return min(1200, max(200, len(text.split()) * 36))

def build_prompt(text, voice):
    tok = llm.tokenize(f'{voice}: {text}'.encode(), add_bos=False, special=False)
    return [128000, 128259] + tok + [128260]

def parse_frames(raw):
    name_nums = [int(m) for m in re.findall(r'<custom_token_(\d+)>', raw)]
    audio = [n for n in name_nums if n >= 10]
    n_full = (len(audio) // 7) * 7
    frames = []
    for i in range(0, n_full, 7):
        frame, valid = [], True
        for pos in range(7):
            code = (audio[i + pos] - 10) - POS_OFFSETS[pos]
            if not (0 <= code < 4096):
                valid = False; break
            frame.append(code)
        if valid:
            frames.append(frame)
    return frames

def frames_to_audio(frames):
    if not frames: return None
    c0 = torch.tensor([f[0] for f in frames], dtype=torch.long).unsqueeze(0)
    c1 = torch.tensor([v for f in frames for v in [f[1], f[4]]], dtype=torch.long).unsqueeze(0)
    c2 = torch.tensor([v for f in frames for v in [f[2], f[3], f[5], f[6]]], dtype=torch.long).unsqueeze(0)
    with torch.inference_mode():
        audio = snac_model.decode([c0.to(DEVICE), c1.to(DEVICE), c2.to(DEVICE)])
    return audio.squeeze().cpu().numpy()

def generate_speech(text, voice='tara', output_path='orpheus_output.wav',
                    max_tokens=None, temperature=0.6, top_p=0.9, rep_penalty=1.3):
    if max_tokens is None:
        max_tokens = calc_max_tokens(text)
    print(f'\n[{voice}] "{text[:60]}"')
    print(f'Tokens: {max_tokens} | Penalty: {rep_penalty} | Running...')
    out = llm.create_completion(
        prompt=build_prompt(text, voice),
        max_tokens=max_tokens, temperature=temperature,
        top_p=top_p, repeat_penalty=rep_penalty,
        stop=['<custom_token_2>'],
    )
    raw, finish = out['choices'][0]['text'], out['choices'][0]['finish_reason']
    frames = parse_frames(raw)
    print(f'Finish: {finish} | Frames: {len(frames)} -> {len(frames)*7} codes')
    if not frames:
        print('FAILED. Raw:', repr(raw[:200]))
        return None
    wav = frames_to_audio(frames)
    sf.write(output_path, wav, samplerate=24000)
    print(f'Saved: {output_path} ({len(wav)/24000:.2f}s)')
    return output_path

print('Functions ready.')

Functions ready.


## Cell 5 - Generate Speech

In [26]:
TEXT  = 'Oh yes, brilliant idea. Let us all work overtime for free again. What a privilege that must be.'
VOICE = 'zeo'
out = generate_speech(TEXT, voice=VOICE, output_path='orpheus_output_zoe_sarcastic.wav')


[zeo] "Oh yes, brilliant idea. Let us all work overtime for free ag"
Tokens: 648 | Penalty: 1.3 | Running...
Finish: length | Frames: 92 -> 644 codes
Saved: orpheus_output_zoe_sarcastic.wav (7.85s)


## Cell 6 - Play Audio

In [ ]:
from IPython.display import Audio, display
if out:
    display(Audio(out, autoplay=True))
else:
    print('No audio.')